# Session 06 Topic 02: Sampling variation and standard error

Use this notebook while working through Topic 02.

Topic 01 ended with a claim: different samples give different means. This notebook
makes that visible, then measures it.

We have a rare luxury here. The 202 Sandringham weekdays of 2023-24 are **all** in the
file, so we know the true population mean. That lets us do something normally
impossible — take samples and check the answers against the truth.

The simulation and charting code is supplied and commented. Read it, run it, and
concentrate on what the results mean. You will write the short calculations and answer
the activity questions.

## 1. Setup

Run this cell. It loads the data and filters to the group used throughout this
session, exactly as you did in Topic 01.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="colorblind")

DATA_FOLDER = Path("data")
TRAIN_FILE = DATA_FOLDER / "train_daily_boardings.csv"

train = pd.read_csv(TRAIN_FILE, parse_dates=["business_date"])

# The same filter as Topic 01: one line, ordinary weekdays, one financial year.
sandringham = train[
    (train["line_name"] == "Sandringham")
    & (train["day_type"] == "Normal Weekday")
    & (train["financial_year"] == "2023-24")
]

# Every one of the 202 days. We are treating this as the whole population.
population = sandringham["total_boardings"]
population_mean = population.mean()

print("Days in the population:", len(population))
print("True population mean:", round(population_mean))

## 2. One sample, one mean

Take 30 of those 202 days at random and calculate their mean.

`random_state` fixes the random choice so that everyone gets the same 30 days. Without
it you would get a different answer every time you ran the cell.

In [ ]:
# Pick 30 days at random from the 202.
#
# replace=True lets the same day be picked more than once. That may look odd, but it
# matches how Topic 01 described the population: not "these 202 days" but "the process
# that produces days like these". Each draw is an independent go at that process.
#
# random_state fixes the result so everyone gets the same 30 days.
one_sample = population.sample(n=30, replace=True, random_state=1106)

print("Population mean (all 202 days):", round(population_mean))
print("Sample mean (30 days):         ", round(one_sample.mean()))
print("Difference:                    ", round(one_sample.mean() - population_mean))

The sample mean is near the population mean but does not match it. Nothing has been
measured incorrectly — the days simply differ from one another, so any 30 of them will
reflect that.

The chart below shows why. The blue bars are all 202 days. The orange ticks along the
bottom mark the 30 days that happened to be picked.

In [ ]:
plt.figure(figsize=(9, 4))

# All 202 days, as a histogram.
sns.histplot(population, bins=24, color="#0072B2", alpha=0.35, label="All 202 weekdays")

# The 30 sampled days, as small ticks along the bottom.
sns.rugplot(one_sample, height=0.08, color="#D55E00", linewidth=2, label="Our sample of 30")

# A dashed line at the true population mean.
plt.axvline(population_mean, color="#666666", linestyle="--", linewidth=2)

plt.title("A sample of 30 weekdays drawn from all 202")
plt.xlabel("Daily boardings")
plt.ylabel("Number of days")
plt.legend()
plt.show()

Now change the sample and watch the answer move. This is a short loop — write it
yourself.

In [ ]:
# Loop over three different random_state values, for example 1, 2 and 3.
# For each one, take a sample of 30 days from population,
#   using replace=True and random_state=seed.
# Print the seed and the mean of that sample.

### Activity — Shift the seed


Run the sample with three different `random_state` values and record the mean for each. How far
apart are your highest and lowest results?

**Your answer:** Double-click this cell and replace this text with your response.

## 3. The sampling distribution

Doing that three times suggests a pattern. Doing it two thousand times shows one.

The loop below takes 2,000 separate 30-day samples and stores each sample's mean.
Every value in `sample_means` is a legitimate estimate of the population mean.

In [ ]:
# An empty list to collect the results.
sample_means = []

# Take 2,000 samples. Each time, store that sample's mean.
#
# Using random_state=i gives every sample its own seed, so the whole simulation
# reproduces exactly. Without it you would get different numbers on every run and
# could not check your answers against anyone else's.
for i in range(2000):
    draw = population.sample(n=30, replace=True, random_state=i)
    sample_means.append(draw.mean())

# Turn the list into a pandas Series so we can summarise it easily.
sample_means = pd.Series(sample_means)

print("Number of sample means collected:", len(sample_means))
print("Smallest:", round(sample_means.min()))
print("Largest: ", round(sample_means.max()))
print("Average of the sample means:", round(sample_means.mean()))

Look at that last line. The average of the 2,000 sample means is almost exactly the
true population mean. The sample mean is an **unbiased** estimator — it does not
systematically run high or low.

Now compare the spread of individual days with the spread of the sample means.

In [ ]:
plt.figure(figsize=(9, 4))

# Individual days: wide and lumpy.
sns.histplot(population, bins=24, color="#0072B2", alpha=0.30, stat="density",
             label="Individual days")

# Means of samples of 30: narrow and bell-shaped.
sns.histplot(sample_means, bins=30, color="#D55E00", alpha=0.75, stat="density",
             label="Means of samples of 30")

plt.axvline(population_mean, color="#666666", linestyle="--", linewidth=2)

plt.title("Individual days vary a lot; their sample means vary far less")
plt.xlabel("Daily boardings")
plt.ylabel("Density")
plt.legend()
plt.show()

Two things to take from this chart.

**The sample means cluster tightly around the true value.** A single Monday might sit
20,000 below a single Thursday, but the mean of 30 mixed days can hardly stray far,
because high and low days cancel each other out within every sample.

**The sample means form a bell shape** — even though the individual days clearly do
not. Section 5 explains why that happens and why it matters so much.

## 4. The standard error

The spread of that orange distribution has a name: the **standard error of the mean**.

You do not normally have 2,000 samples to work from. Fortunately it can be estimated
from a single sample, using the formula

$$SE = \frac{s}{\sqrt{n}}$$

where $s$ is the sample standard deviation and $n$ is the sample size.

In [ ]:
# Calculate the standard deviation of one_sample.
# Divide it by the square root of 30 to get the standard error.
#   np.sqrt() takes a square root.
# Print both values.

Check the estimate against the truth. The next cell measures the actual spread of the
2,000 sample means — something you could only do because we hold the whole population.

In [ ]:
print("Standard error estimated from ONE sample of 30:", round(standard_error))
print("Actual spread of the 2,000 sample means:      ", round(sample_means.std()))

The two are close. That is the point of the formula: **one sample is enough** to
estimate how much the answer would vary if you sampled again.

Keep the two measures apart. They answer different questions.

| | Question it answers |
|---|---|
| **Standard deviation** ($s$) | how much do individual **days** differ from each other? |
| **Standard error** ($s/\sqrt{n}$) | how much would the **mean** differ if I sampled again? |

The $\sqrt{n}$ in the denominator has a consequence worth pinning down now, because
Topic 03 builds on it.

To see the effect of sample size on its own, the next cell holds the **spread** fixed
and varies only $n$. (In real work $s$ is estimated from each sample and wobbles a
little, which would blur the pattern.)

In [ ]:
# Store the standard deviation of population as spread.
# Loop over the sample sizes 30, 60, 120 and 480.
# For each one, divide spread by the square root of n and print the result.

### Activity — Predict, then check


Before running the cell above, predict what happens to the standard error if the sample
size rises from 30 to 120. Then calculate both and see whether you were right.

**Your answer:** Double-click this cell and replace this text with your response.

## 5. Why does the bell shape appear

The sample means formed a bell curve even though daily boardings do not. That is the
**central limit theorem**:

> If you take random samples of a reasonable size from *any* population, the
> distribution of the sample means approaches a normal (bell-shaped) distribution,
> regardless of the shape of the original population.

There is no sample size that guarantees this for every population. **n = 30** is a
common classroom rule of thumb, but strongly skewed data or extreme outliers may require a
larger sample. If the population is already close to normal, smaller samples can also
produce an approximately normal sampling distribution.

## 6. Reading the normal curve

A normal distribution is symmetric about its mean, and the area under any part of the
curve gives the proportion of values falling there.

| Range | Proportion of values inside |
|---|---|
| within 1 standard deviation of the mean | about 68% |
| within 1.96 standard deviations | **95%** |
| within 2.58 standard deviations | about 99% |

![Normal curve with the main probability areas shaded. The ±2 labels approximate the
exact ±1.96 values used for 95%.](data/s06-t02-normal-curve-z-areas.png)

The middle row is used constantly from here on. **1.96** is simply how far you must go
either side of the centre to capture 95% of a normal distribution.

In [ ]:
from scipy.stats import norm

# norm.ppf() answers: "how many standard deviations below this point lies X% of the curve?"
# 0.975 leaves 2.5% in the upper tail, and by symmetry 2.5% in the lower tail,
# so 95% sits in the middle.
print("Value that leaves 2.5% in the upper tail:", round(norm.ppf(0.975), 4))

### Activity — Locate the tails


If 95% of values sit between −1.96 and +1.96, what percentage sits above +1.96? Sketch
the curve and shade the two tails.

**Your answer:** Double-click this cell and replace this text with your response.

## 7. Turning it around

Section 6 describes where sample means fall **when you already know the population
mean** — which you never do. The useful move is to reverse the statement:

> If a sample mean is within 1.96 standard errors of the population mean 95% of the
> time, then an interval stretching 1.96 standard errors either side of **your** sample
> mean will contain the population mean 95% of the time.

The simulation below takes 20 samples, builds that interval for each, and checks
whether it caught the true mean.

In [ ]:
# Collect one interval per sample.
results = []

for i in range(20):
    # random_state=i makes every interval reproducible.
    draw = population.sample(n=30, replace=True, random_state=i)

    mean = draw.mean()
    standard_error = draw.std() / np.sqrt(30)

    # Stretch 1.96 standard errors either side of the sample mean.
    low = mean - 1.96 * standard_error
    high = mean + 1.96 * standard_error

    # Did this interval capture the true population mean?
    caught = low <= population_mean <= high

    results.append({"sample": i, "mean": mean, "low": low, "high": high, "caught": caught})

results = pd.DataFrame(results)

print("Intervals that contained the true mean:", results["caught"].sum(), "out of 20")

In [ ]:
plt.figure(figsize=(9, 5))

# Draw one horizontal line per interval. Orange if it missed the true mean.
for row in results.itertuples():
    colour = "#0072B2" if row.caught else "#D55E00"
    plt.plot([row.low, row.high], [row.sample, row.sample], color=colour, linewidth=2.5)
    plt.plot(row.mean, row.sample, "o", color=colour, markersize=6)

# The true population mean never moves. The intervals do.
plt.axvline(population_mean, color="#666666", linestyle="--", linewidth=2)

plt.title("Twenty samples, twenty intervals")
plt.xlabel("Daily boardings")
plt.ylabel("Sample number")
plt.yticks([])
plt.show()

Read this chart carefully, because it is the idea the rest of the session rests on.

**The dashed line never moves.** The population mean is a fixed number. What varies is
the intervals — one per sample, jumping around it. Here 18 of the 20 caught it and two
missed.

**The intervals that missed look no different from the others.** There is no way to
tell from the inside whether yours is one of the successes or one of the failures.
That is the honest cost of working from a sample, and it is exactly why the phrase is
"95% confident" rather than "certain".

Note that 18 out of 20 is 90%, not 95%. With only 20 intervals that is ordinary luck —
you would expect one miss on average, and two is unremarkable.

The guarantee is about the long run, so test it over a long run. The next cell repeats
the same procedure 2,000 times and counts.

In [ ]:
caught_count = 0

# Exactly the same steps as above, just repeated many more times.
for i in range(2000):
    draw = population.sample(n=30, replace=True, random_state=i)

    mean = draw.mean()
    standard_error = draw.std() / np.sqrt(30)

    low = mean - 1.96 * standard_error
    high = mean + 1.96 * standard_error

    if low <= population_mean <= high:
        caught_count = caught_count + 1

print("Intervals that contained the true mean:", caught_count, "out of 2000")
print("That is", round(caught_count / 2000 * 100, 1), "percent")

**95.0%.** The method does what it claims — not for any particular interval, but
across many.

That is the precise meaning of "95% confident", and it is worth holding on to as
Topic 03 turns this into something you report to a client.

## What you have done

- Watched a single sample mean miss the true value, and seen why that is normal
- Built a **sampling distribution** from 2,000 samples, and found it centred on the
  truth and far narrower than the raw data
- Estimated the **standard error** from one sample and checked it against the real
  spread
- Confirmed that four times the data halves the standard error, not quarters it
- Found where **1.96** comes from
- Built intervals from 20 samples, then from 2,000, and confirmed that 95% of them
  caught the true mean

**Next:** Topic 03's notebook turns this into a confidence interval you can report.

> One detail to carry forward: we used **1.96** here, which is exact when the
> population standard deviation is known. Working from a sample it is slightly too
> small, and Topic 03 replaces it with a marginally larger value from the
> t-distribution.